# nb00 Data Probe: DMHC plan financial sources

**Purpose**
- Probe each DMHC (Department of Managed Health Care) source to learn what it actually serves: page structure, plan identifiers, table layouts, export options.
- Save raw snapshots to `data/raw/probe/` and print diagnostics; nothing is cleaned or extracted yet.
- The extract step gets designed from this run's printed output.

**Prerequisites**
- Python 3 with `requests`, `beautifulsoup4`, and `pdfplumber` installed (`pip install requests beautifulsoup4 pdfplumber`).
- Run from the `tableau_plan_financials/notebooks/` folder.
- Internet access to dmhc.ca.gov and wpso.dmhc.ca.gov.

**Outputs**
- HTML and PDF snapshots under `data/raw/probe/`.
- Diagnostics in `nb00_data_probe_cell_output.txt`; every probe is wrapped so one failing source never stops the others.

In [1]:
# Step 0: mirror all printed output to a text file for easy sharing
import sys
from pathlib import Path

SINK_PATH = Path.cwd() / "nb00_data_probe_cell_output.txt"

_orig_out = getattr(sys, "_nb_orig_stdout", sys.stdout)
_orig_err = getattr(sys, "_nb_orig_stderr", sys.stderr)
sys._nb_orig_stdout, sys._nb_orig_stderr = _orig_out, _orig_err

class _Tee:
    def __init__(self, stream, fh):
        self.stream, self.fh = stream, fh
    def write(self, data):
        self.stream.write(data)
        self.fh.write(data)
        self.fh.flush()
    def flush(self):
        self.stream.flush()
        self.fh.flush()

_sink = open(SINK_PATH, "w")
sys.stdout = _Tee(_orig_out, _sink)
sys.stderr = _Tee(_orig_err, _sink)
print(f"Mirroring cell output to {SINK_PATH.name} (attach this file in the chat)")

Mirroring cell output to nb00_data_probe_cell_output.txt (attach this file in the chat)
Probe helpers ready. Snapshots -> /Users/trinidadcisneros/Documents/Development/trinidadcisneros.github.io/folders/ds_blogs/projects/tableau/tableau_plan_financials/data/raw/probe
[hpsearch_viewall] status 200, 153,530 bytes, final url https://wpso.dmhc.ca.gov/hpsearch/viewall.aspx
links on page: 164; candidate plan links containing 'care'/'health net': 30
  AIDS Healthcare FoundationFull Service                         -> details.aspx?id=933 0432&name=&page=all
  Align Senior Care California, Inc.Full Service                 -> details.aspx?id=933 0554&name=&page=all
  AltaMed Health Network, Inc.Full Service                       -> details.aspx?id=933 0492&name=&page=all
  Bay Area Accountable Care Network, Inc.Full Service            -> details.aspx?id=933 0519&name=&page=all
  BZ Health Network of California, Inc.Full Service              -> details.aspx?id=933 0506&name=&page=all
  Carelon Beh

In [2]:
# Step 1: setup
from pathlib import Path
import requests
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd().parent
PROBE_DIR = PROJECT_ROOT / "data" / "raw" / "probe"
PROBE_DIR.mkdir(parents=True, exist_ok=True)

S = requests.Session()
S.headers.update({"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X) research probe for a public data blog"})

def probe(name, url, binary=False):
    """Fetch a URL, save the snapshot, return (response, soup or None). Never raises."""
    try:
        r = S.get(url, timeout=60, allow_redirects=True)
        ext = "pdf" if binary else "html"
        out = PROBE_DIR / f"{name}.{ext}"
        out.write_bytes(r.content)
        print(f"[{name}] status {r.status_code}, {len(r.content):,} bytes, final url {r.url}")
        if binary:
            return r, None
        return r, BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"[{name}] FAILED: {type(e).__name__}: {e}")
        return None, None

print("Probe helpers ready. Snapshots ->", PROBE_DIR)

In [3]:
# Step 2: the plan directory, to find the two license identifiers
r, soup = probe("hpsearch_viewall", "https://wpso.dmhc.ca.gov/hpsearch/viewall.aspx")
if soup:
    rows = [a for a in soup.find_all("a") if a.get("href")]
    hits = [a for a in rows if "care" in a.get_text(strip=True).lower() or "health net" in a.get_text(strip=True).lower()]
    print(f"links on page: {len(rows)}; candidate plan links containing \'care\'/\'health net\': {len(hits)}")
    for a in hits[:40]:
        print(f"  {a.get_text(strip=True)[:60]:62s} -> {a['href']}")

In [4]:
# Step 3: the financial summary application
r, soup = probe("flash_app", "http://wpso.dmhc.ca.gov/flash/")
if soup:
    print("title:", soup.title.get_text(strip=True) if soup.title else "none")
    for f in soup.find_all("form"):
        print("form action:", f.get("action"), "| method:", f.get("method"))
    selects = soup.find_all("select")
    print(f"dropdowns: {len(selects)}")
    for s in selects[:4]:
        opts = s.find_all("option")
        print(f"  select name={s.get('name')} id={s.get('id')} options={len(opts)}")
        for o in opts[:5]:
            print(f"    value={o.get('value')} text={o.get_text(strip=True)[:50]}")
    for a in soup.find_all("a")[:20]:
        print("  link:", a.get_text(strip=True)[:40], "->", a.get("href"))

In [5]:
# Step 4: the financial stability dashboard
r, soup = probe("finances_dashboard", "https://wpso.dmhc.ca.gov/dashboard/Finances.aspx")
if soup:
    print("title:", soup.title.get_text(strip=True) if soup.title else "none")
    selects = soup.find_all("select")
    print(f"dropdowns: {len(selects)}")
    for s in selects:
        opts = s.find_all("option")
        sample = [o.get_text(strip=True)[:40] for o in opts if "l.a. care" in o.get_text(strip=True).lower()
                  or "health net" in o.get_text(strip=True).lower()]
        print(f"  select name={s.get('name')} options={len(opts)} matches={sample[:4]}")
    print("has __VIEWSTATE:", bool(soup.find(id="__VIEWSTATE")))
    tables = soup.find_all("table")
    print(f"tables on landing: {len(tables)}")

In [6]:
# Step 5: the financial statements eFiling search
r, soup = probe("fe_search", "https://wpso.dmhc.ca.gov/fe/search/")
if soup:
    print("title:", soup.title.get_text(strip=True) if soup.title else "none")
    for f in soup.find_all("form"):
        print("form action:", f.get("action"), "| method:", f.get("method"))
    for inp in soup.find_all(["input", "select"])[:15]:
        print(f"  {inp.name} name={inp.get('name')} id={inp.get('id')} type={inp.get('type')}")

In [7]:
# Step 6: the FSSB Medi-Cal plan summary PDF, the fallback source
r, _ = probe("fssb_aug25", "https://www.dmhc.ca.gov/Portals/0/Docs/OFR/FSSB/Aug25/FinancialSummaryofMediCalManagedHealthPlans.pdf", binary=True)
if r is not None:
    try:
        import pdfplumber
        with pdfplumber.open(PROBE_DIR / "fssb_aug25.pdf") as pdf:
            print(f"pages: {len(pdf.pages)}")
            for i, page in enumerate(pdf.pages[:6]):
                tables = page.extract_tables()
                text_head = (page.extract_text() or "")[:200].replace("\n", " | ")
                print(f"  page {i+1}: {len(tables)} tables | {text_head}")
                for t in tables[:1]:
                    for row in t[:4]:
                        print("    ", [str(c)[:22] if c else "" for c in row])
    except ImportError:
        print("pdfplumber not installed; run: pip install pdfplumber, then rerun this cell")

## Stage 2: submit the forms (added after the first probe run)

The first run confirmed every source is reachable from this machine and settled three things:

- License identifiers from the plan directory:
  - L.A. Care Health Plan Joint Powers Authority: `933 0504`
  - Health Net Community Solutions, Inc.: `933 0426`
- The Financial Summary application is a plain ASP.NET POST form (plan listbox keyed by license id, report type dropdown), so it can be submitted directly.
- The FSSB PDF is a 4 page slide deck with zero extractable tables, so it is dropped as a data fallback.

Stage 2 submits the forms and prints what comes back: table shapes, sample rows, and any file links. Still diagnostics only.

**Added after the second run.** The dropdown dump revealed L.A. Care holds TWO licenses:

- `933 0355` Local Initiative Health Authority for Los Angeles County (the dashboard labels it "L.A. Care Health Plan"), and
- `933 0504` L.A. Care Health Plan Joint Powers Authority.

Step 7b probes both license detail pages to establish which one carries the Medi-Cal line of business. Step 8 now submits the report form correctly: the first attempt came back as the unchanged form because no measure checkboxes were ticked and no date range was set.

In [8]:
# Step 7: pin down the exact dropdown entry for each plan in each application
TARGET_WORDS = ["care", "health net", "local", "los angeles"]

def dump_plan_options(sel, label):
    opts = sel.find_all("option")
    print(f"[{label}] total options: {len(opts)}; entries containing any of {TARGET_WORDS}:")
    for o in opts:
        t = o.get_text(strip=True)
        if any(w in t.lower() for w in TARGET_WORDS):
            print(f"    value={o.get('value')!r:16} text={t[:70]}")

r_flash, soup_flash = probe("flash_app_stage2", "https://wpso.dmhc.ca.gov/flash/")
if soup_flash:
    sel = soup_flash.find("select", id="MainContent_MainContent_lbHP")
    if sel:
        dump_plan_options(sel, "summary app lbHP")
    print("all form inputs on the summary app (needed to build a faithful POST):")
    for inp in soup_flash.find_all("input"):
        v = (inp.get("value") or "")[:25]
        print(f"    input name={inp.get('name')} type={inp.get('type')} value={v!r}")

r_dash, soup_dash = probe("finances_dashboard_stage2", "https://wpso.dmhc.ca.gov/dashboard/Finances.aspx")
if soup_dash:
    sel = soup_dash.find("select", attrs={"name": "ctl00$MainContent$dmhcSelectHP$ddlHealthPlans"})
    if sel:
        dump_plan_options(sel, "dashboard ddlHealthPlans")

r_fe, soup_fe = probe("fe_search_stage2", "https://wpso.dmhc.ca.gov/fe/search/")
if soup_fe:
    sel = soup_fe.find("select", id="MainContent_MainContent_ddlHP")
    if sel:
        dump_plan_options(sel, "eFiling search ddlHP")

In [9]:
# Step 7b: which L.A. Care license is which? Probe both license detail pages
for tag, lic in [("lacare_local_initiative", "933 0355"), ("lacare_jpa", "933 0504")]:
    r, soup = probe(f"hpsearch_{tag}", f"https://wpso.dmhc.ca.gov/hpsearch/details.aspx?id={lic}&name=&page=all")
    if soup:
        shown = 0
        for t in soup.find_all("table")[:6]:
            for row in t.find_all("tr")[:25]:
                cells = [c.get_text(strip=True)[:45] for c in row.find_all(["th", "td"])]
                if any(cells):
                    print("    ", cells[:6])
                    shown += 1
        for dt in soup.find_all("dt")[:25]:
            dd = dt.find_next_sibling("dd")
            print(f"     {dt.get_text(strip=True)[:40]}: {dd.get_text(strip=True)[:60] if dd else ''}")
            shown += 1
        if not shown:
            text = soup.get_text(" ", strip=True)
            k = text.lower().find("license")
            print("     text sample:", text[max(0, k - 100):k + 900] if k >= 0 else text[:900])

In [10]:
# Step 8: submit the Financial Summary form once per license and describe what comes back
# The first attempt returned the form unchanged: no measure checkboxes were ticked and no
# date range was set. This version checks every financial measure, the two enrollment
# measures the per member month math needs, and a wide date range.
def build_payload(form):
    """Collect every input and select on the form with its default value, ASP.NET style."""
    payload = {}
    for inp in form.find_all("input"):
        name = inp.get("name")
        if not name:
            continue
        itype = (inp.get("type") or "text").lower()
        if itype in ("submit", "button", "image"):
            continue  # only the clicked button gets added, by the caller
        if itype in ("checkbox", "radio") and not inp.has_attr("checked"):
            continue
        payload[name] = inp.get("value") or ""
    for sel in form.find_all("select"):
        name = sel.get("name")
        if not name:
            continue
        opt = sel.find("option", selected=True) or sel.find("option")
        payload[name] = (opt.get("value") if opt else "") or ""
    return payload

def describe_response(name, r):
    """Save a POST response snapshot and print its structure: tables, sample rows, file links."""
    soup = BeautifulSoup(r.text, "html.parser")
    (PROBE_DIR / f"{name}.html").write_bytes(r.content)
    print(f"[{name}] status {r.status_code}, {len(r.content):,} bytes, final url {r.url}")
    print("  title:", soup.title.get_text(strip=True) if soup.title else "none")
    tables = soup.find_all("table")
    print(f"  tables: {len(tables)}")
    for t in tables[:6]:
        rows = t.find_all("tr")
        print(f"    table with {len(rows)} rows; first rows:")
        for row in rows[:8]:
            cells = [c.get_text(strip=True)[:24] for c in row.find_all(["th", "td"])]
            print("     ", cells[:10])
    file_links = [a for a in soup.find_all("a")
                  if a.get("href") and any(x in a["href"].lower() for x in (".pdf", ".xls", ".csv", "download"))]
    print(f"  file-like links: {len(file_links)}")
    for a in file_links[:12]:
        print("   ", a.get_text(strip=True)[:50], "->", a["href"][:95])
    return soup

def find_plan_value(soup, select_id, needles, fallback):
    """Read the plan's option value out of the live dropdown; fall back to the directory id."""
    sel = soup.find("select", id=select_id)
    if sel:
        for o in sel.find_all("option"):
            t = o.get_text(strip=True).lower()
            if any(n in t for n in needles):
                print(f"  matched option: value={o.get('value')!r} text={o.get_text(strip=True)[:70]}")
                return o.get("value")
    print(f"  no dropdown match; falling back to {fallback!r}")
    return fallback

FLASH_URL = "https://wpso.dmhc.ca.gov/flash/"
# The Create Report button onclick posts the form to flash.aspx in a new window;
# posting to the landing URL only re-renders the form. The report lives here:
REPORT_URL = "https://wpso.dmhc.ca.gov/flash/flash.aspx"
P = "ctl00$ctl00$MainContent$MainContent$"

MEASURES = ["tne", "req_tne", "excess", "tne_required", "totalAssets", "totalCurrentAssets",
            "totalCurrentLiabilities", "revenue", "income_loss", "admin_exp", "admin_ratio",
            "med_exp", "medloss_ratio"]
ENROLLMENT = {"0": "Total Enrollees", "1": "Medi-Cal Managed Care"}

def flash_submit(tag, needles, fallback_id):
    try:
        r0 = S.get(FLASH_URL, timeout=60)
        soup0 = BeautifulSoup(r0.text, "html.parser")
        form = soup0.find("form")
        plan_value = find_plan_value(soup0, "MainContent_MainContent_lbHP", needles, fallback_id)
        payload = build_payload(form)
        payload[P + "lbHP"] = plan_value
        payload[P + "lbHPType"] = "0"
        payload[P + "ddlReportType"] = "0"
        # Dates MUST be month name plus year (a numeric date crashes the report engine),
        # and a 26 year window times out server side; 6 years works.
        payload[P + "txtStartDate"] = "July 2020"
        payload[P + "txtEndDate"] = "July 2026"
        for i, v in enumerate(MEASURES):
            payload[f"{P}cblFinancial${i}"] = v
        for i, v in ENROLLMENT.items():
            payload[f"{P}cblEnrollment${i}"] = v
        payload[P + "cmdSubmit"] = "Create Report"
        r1 = S.post(REPORT_URL, data=payload, timeout=180)
        describe_response(f"flash_post_{tag}", r1)
    except Exception as e:
        print(f"[flash_post_{tag}] FAILED: {type(e).__name__}: {e}")

flash_submit("lacare_local_initiative", ["local initiative"], "933 0355")
flash_submit("lacare_jpa", ["joint powers"], "933 0504")
flash_submit("hncs", ["health net community"], "933 0426")

In [11]:
# Step 9: the eFiling search, one attempted search for L.A. Care
FE_URL = "https://wpso.dmhc.ca.gov/fe/search/"
try:
    r0 = S.get(FE_URL, timeout=60)
    soup0 = BeautifulSoup(r0.text, "html.parser")
    form = soup0.find("form")
    sel = soup0.find("select", id="MainContent_MainContent_ddlHP")
    opts = sel.find_all("option") if sel else []
    print(f"[fe_search_post] ddlHP options on first load: {len(opts)}")
    stype = soup0.find("select", id="MainContent_MainContent_ddlStatementType")
    if stype:
        for o in stype.find_all("option"):
            print(f"  statement type: value={o.get('value')!r} text={o.get_text(strip=True)[:60]}")
    plan_val = None
    for o in opts:
        t = o.get_text(strip=True).lower()
        if "l.a. care" in t or "local initiative" in t:
            plan_val = o.get("value")
            print(f"  matched option: value={plan_val!r} text={o.get_text(strip=True)[:70]}")
            break
    if plan_val:
        payload = build_payload(form)
        payload["ctl00$ctl00$MainContent$MainContent$ddlHP"] = plan_val
        payload["ctl00$ctl00$MainContent$MainContent$btnSearch"] = "Search"
        payload.pop("ctl00$ctl00$MainContent$MainContent$btnReset", None)
        r1 = S.post(FE_URL, data=payload, timeout=120)
        describe_response("fe_search_post_lacare", r1)
    else:
        print("  no L.A. Care option on first load; the plan list likely fills in after a plan-type postback.")
        print("  Deferring: only worth automating if the summary application falls short.")
except Exception as e:
    print(f"[fe_search_post] FAILED: {type(e).__name__}: {e}")

In [12]:
# Step 10: confirm the output sink
sys.stdout.flush()
print(f"\nAll printed output saved to: {SINK_PATH}")
print(f"File size: {SINK_PATH.stat().st_size:,} bytes")

**Next step**
- Attach `nb00_data_probe_cell_output.txt` in the chat.
- Stage 2's printed tables and file links decide the extract design: the summary application response if it serves parseable tables, otherwise the eFiling PDFs.
- The FSSB August 2025 PDF turned out to be a 4 page slide deck with no tables; it stays as narrative context only, not a data source.